# Synthetic data design — GT images & how the ground truth is built

This notebook visualises the **synthetic generator** (`scripts/make_realistic_cases.py` +
`docvlm_eval.synth`). For each case it shows the **GT image** and **how its ground truth is
designed** — with emphasis on the part that matters most: the **non-OCR understanding GT**.

The OCR GT (what string is where) falls out of the render for free. The valuable, harder GT —
*where is this word / table?*, *how many times does it appear?*, *what is the total?* and the
**reasoning** behind each — is derived **with no external model** (`docvlm_eval.synth.derive`),
purely from the rendered PDF's exact text positions + the values the generator already knows.
Every answer is therefore gold by construction.

Run order: Runtime → Run all. (Needs the `[synth]` extra: weasyprint + pymupdf + faker.)

In [ ]:
# --- install docvlm_eval (fresh env e.g. Colab: clone+checkout; then editable install) ---
import os, sys, subprocess, importlib
from pathlib import Path

def _repo_root():
    for c in (Path.cwd(), Path.cwd().parent):
        if (c / "pyproject.toml").exists():
            return c
    return None

root = _repo_root()
if root is None:                       # fresh environment (Colab/Kaggle): clone the repo
    subprocess.run(["git", "clone", "https://github.com/SangbumChoi/OCR.git"], check=False)
    root = Path("OCR")
    subprocess.run(["git", "-C", str(root), "checkout", "claude/new-session-w79q0i"], check=False)
    subprocess.run(["git", "-C", str(root), "pull", "--ff-only"], check=False)
os.chdir(root)                         # cwd is now the repo root
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[synth]"], check=True)

# The editable-install .pth is only read at interpreter startup, so a running kernel can't import
# the package until we add src/ to sys.path ourselves (avoids "No module named docvlm_eval").
src = str(Path.cwd() / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()
import docvlm_eval
print("docvlm_eval ready from", docvlm_eval.__file__)


## 0. Setup + (re)generate the cases with the generator

In [ ]:
import sys, json, subprocess
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

ROOT = Path.cwd()
if (ROOT / "scripts").exists() is False and (ROOT.parent / "scripts").exists():
    ROOT = ROOT.parent            # tolerate being launched from notebooks/
sys.path.insert(0, str(ROOT / "src"))
CASES_DIR = ROOT / "data" / "probes" / "realistic_cases"

# generate fresh (clean only = fast); each gt.json is the structured DocSample DTO
subprocess.run([sys.executable, "scripts/make_realistic_cases.py", "--no-degrade"], cwd=ROOT, check=True)
print("cases:", sorted(p.name for p in CASES_DIR.iterdir() if p.is_dir()))

## 1. Helper — overlay GT boxes and list the *designed* GT

`fields_detailed[].bbox` = OCR/KIE spotting boxes (green). The **understanding** layer adds, in
distinct colours: **locate** (a word's box, red), **region** (a table/region box, blue). Below each
image we print the derived understanding QAs — *question → answer → reasoning* — so you can see
exactly how the GT was designed.

In [ ]:
UNDERSTANDING = {"L1-locate", "L1-region", "H-count", "H1-aggregate"}
COLOR = {"field": "#1a9641", "L1-locate": "#d7191c", "L1-region": "#2c7bb6"}

def load_case(key):
    d = json.loads((CASES_DIR / key / "gt.json").read_text())
    img = Image.open(CASES_DIR / key / "clean.png").convert("RGB")
    return d, img

def show_case(key):
    """One full card per case: the GT image with overlays, the document text it contains, and EVERY
    task it defines (question -> answer -> reasoning), so it's clear what can be learned/evaluated."""
    gt, img = load_case(key)
    fig, ax = plt.subplots(figsize=(7, 7 * img.height / img.width))
    ax.imshow(img); ax.set_axis_off()
    ax.set_title(f"{key}  ·  {gt.get('type','')}", fontsize=11)

    def rect(b, color, label):
        ax.add_patch(mpatches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                     fill=False, edgecolor=color, linewidth=1.8))
        ax.text(b[0], max(0, b[1]-4), label, color=color, fontsize=7,
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.6, pad=0.5))
    for f in gt.get("fields_detailed", []):
        if f.get("bbox"):
            rect(f["bbox"], COLOR["field"], f["key"])                 # field / KIE spot (green)
    for q in gt.get("qa_detailed", []):
        if q.get("answer_bbox") and q["answer_type"] in ("L1-locate", "L1-region"):
            rect(q["answer_bbox"], COLOR[q["answer_type"]],
                 "locate" if q["answer_type"] == "L1-locate" else "region")  # red / blue
    plt.tight_layout(); plt.show()

    print(f"TYPE      : {gt.get('type','')}   domain={gt.get('domain')}  "
          f"acquisition={gt.get('acquisition')}  langs={gt.get('languages')}")
    # what document TEXT/INFO is on the page (the OCR-side GT)
    flds = [f for f in gt.get("fields_detailed", []) if f.get("value")]
    if flds:
        print("DOCUMENT TEXT (fields):")
        for f in flds:
            v = f["value"] if len(f["value"]) <= 48 else f["value"][:48] + "…"
            tags = []
            if f.get("language", "en") != "en": tags.append(f"{f['language']}/{f.get('script')}")
            if f.get("is_small"): tags.append("small-text")
            if f.get("bbox"): tags.append("box")
            print(f"   - {f['key']:18}: {v!r}  {('· ' + ' '.join(tags)) if tags else ''}")
    # what TASKS the case defines (every QA + control probe), with answer + reasoning
    print("TASKS (what a model is asked to do):")
    for q in gt.get("qa_detailed", []):
        mark = "★" if q["answer_type"] in UNDERSTANDING else " "
        print(f"  {mark}[{q['answer_type']:12}] {q['question']}")
        print(f"        → answer: {q['answers'][0]}   (metric={q['metric']})")
        if q.get("rationale"):
            print(f"        → reasoning: {q['rationale']}")
    for pr in gt.get("probes", []):
        print(f"   [probe:{pr['kind']:9}] {pr['question']}   → expect: {pr['expected']}")
    if gt.get("table_html"):
        print(f"   [table       ] convert the table to HTML (TEDS gold, {gt['table_html'][:30]}…)")
    sup = gt.get("ablation_support", {})
    print("SUPPORTS  :", {k: v for k, v in sup.items() if v}, "   (★ = model-free understanding GT)")
    print("=" * 100)

## 2. Gallery — every case drawn like the invoice example

For each generated document: the **GT image with overlays** (green=field/KIE box, red=locate, blue=region), the **document text** it contains, and **every task** it defines — question → answer → *reasoning* (★ marks the model-free understanding GT: locate / region / count / sum).

In [ ]:
# Full gallery: draw every generated case like the invoice example — GT image + tasks + reasoning.
cases = sorted(p.name for p in CASES_DIR.iterdir() if p.is_dir())
print(f"{len(cases)} synthetic cases:", cases, "\n")
for key in cases:
    show_case(key)

## 3. How each GT type is designed (model-free)

| GT type | Question it answers | Derived from | Reasoning it emits |
| --- | --- | --- | --- |
| `L1-locate` | *Where is this word?* | exact text box from the rendered PDF | "'X' is at [x1,y1,x2,y2]" |
| `L1-region` | *Where is this table/region?* | union of the member strings' boxes | spans [..] enclosing the cells |
| `H-count` | *How many times does X appear?* | count of the word's hits on the page | the N hit positions |
| `H1-aggregate` | *What is the total?* | arithmetic over known values | the worked sum `a + b + c = T` |

No external model is used — only PyMuPDF text positions + arithmetic — so all answers are exact.
Each is one `DocBuilder.ask_*` call (`ask_where` / `ask_region` / `ask_count` / `ask_aggregate`).

## 4. Flexibility — toggle the understanding layer as one ablation switch

`emit_understanding` in `configs/synth_data.yaml` (or `--ablation U_understanding_off`) includes or
removes the whole understanding layer, so its contribution can be ablated in isolation.

In [ ]:
for arm in ["U_understanding_on", "U_understanding_off"]:
    subprocess.run([sys.executable, "scripts/make_realistic_cases.py", "--only", "invoice",
                    "--no-degrade", "--ablation", arm], cwd=ROOT, check=True)
    gt = json.loads((CASES_DIR / "invoice" / "gt.json").read_text())
    n = sum(q["answer_type"] in UNDERSTANDING for q in gt.get("qa_detailed", []))
    print(f"{arm:22} -> {n} understanding QAs")
# restore the full baseline
subprocess.run([sys.executable, "scripts/make_realistic_cases.py", "--no-degrade"], cwd=ROOT, check=True)

## 5. At-a-glance index (thumbnails + support flags)

In [ ]:
# Compact at-a-glance index of all cases with their ablation-support flags.
keys = sorted(p.name for p in CASES_DIR.iterdir() if p.is_dir())
cols = 4; rows = (len(keys) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(14, 3.4 * rows))
for ax, key in zip(axes.ravel(), keys):
    gt, img = load_case(key)
    ax.imshow(img); ax.set_axis_off()
    flags = "".join(c for c, on in [("S","spotting"),("R","rationale"),("M","multilingual"),
            ("u","small_text"),("T","table"),("A","abstain")]
            if gt.get("ablation_support", {}).get(on))
    ax.set_title(f"{key}\n[{flags}]", fontsize=8)
for ax in axes.ravel()[len(keys):]:
    ax.set_axis_off()
plt.tight_layout(); plt.show()
print("flags: S=spotting R=rationale M=multilingual u=small-text T=table A=abstain")